# GCC Insurance Market Data Cleaning

This notebook takes the two raw datasets collected in `01_data_collection.ipynb` 
and prepares them for analysis:
- Setup & Load Raw Data
- Standardize country naming across both datasets
- Fix data types (year as integer, not text)
- Select each country's most recent available macro data (UAE falls back 
  to 2024 since 2025 GDP is not yet published; documented per row)
- Merge into a single analysis-ready table
- Re-run data quality checks on the merged result
- Save the cleaned output to `data/cleaned/`

**Inputs:** `data/raw/gcc_insurance_2025.csv`, `data/raw/gcc_worldbank_macro.csv`

**Output:** `data/cleaned/gcc_merged.csv`

### 1. Setup & Load Raw Data



In [1]:
import pandas as pd

INSURANCE_PATH = "data/raw/gcc_insurance_2025.csv"
WORLDBANK_PATH = "data/raw/gcc_worldbank_macro.csv"

gcc_insurance = pd.read_csv(INSURANCE_PATH)
worldbank_macro = pd.read_csv(WORLDBANK_PATH)


assert gcc_insurance.shape == (6, 7), f"Unexpected insurance data shape: {gcc_insurance.shape}"
assert worldbank_macro.shape == (96, 5), f"Unexpected World Bank data shape: {worldbank_macro.shape}"


gcc_insurance["gcc_share_pct"] = (gcc_insurance["gwp_usd_billion"] / gcc_insurance["gwp_usd_billion"].sum() * 100).round(1)

print("Insurance data loaded:", gcc_insurance.shape)
print("World Bank data loaded:", worldbank_macro.shape)

Insurance data loaded: (6, 8)
World Bank data loaded: (96, 5)


### 2. Standardize Country Names

The World Bank dataset uses full country names ("United Arab Emirates"), 
while the insurance dataset uses "UAE". Before these two datasets can be 
merged on country, both must use identical naming. We standardize both to 
each country's common short name, since that's more portable for 
visualizations and presentations later.

In [2]:
print("Insurance data countries:", sorted(gcc_insurance["country"].unique()))
print("World Bank countries:", sorted(worldbank_macro["country"].unique()))

Insurance data countries: ['Bahrain', 'Kuwait', 'Oman', 'Qatar', 'Saudi Arabia', 'UAE']
World Bank countries: ['Bahrain', 'Kuwait', 'Oman', 'Qatar', 'Saudi Arabia', 'United Arab Emirates']


In [3]:
worldbank_macro["country"] = worldbank_macro["country"].replace("United Arab Emirates", "UAE")


print("World Bank countries after fix:", sorted(worldbank_macro["country"].unique()))

World Bank countries after fix: ['Bahrain', 'Kuwait', 'Oman', 'Qatar', 'Saudi Arabia', 'UAE']


### 3. Fix Data Types 

In [4]:
print("Before:", worldbank_macro["year"].dtype)
worldbank_macro["year"] = worldbank_macro["year"].astype(int)
print("After:", worldbank_macro["year"].dtype)

Before: int64
After: int64


Verify Data Types

The `year` column was documented as text in `01_data_collection.ipynb`. 
Reloading from the saved CSV in this notebook, pandas' automatic type 
inference already converts it to integer, since every value is a plain 
number with no special characters. Verified below rather than removed, 
to make this explicit and confirmed rather than assumed.

### 4. Merging Datasets

The insurance dataset is a single-year snapshot (2025), while the World 
Bank data spans 2010–2025 though UAE's 2025 GDP is not yet published 
(see Finding in `01_data_collection.ipynb`).

Two outputs are created:
1. `worldbank_macro` (cleaned, full history) retained as-is for 
   time-series analysis of GDP/population trends
2. `gcc_merged` a cross-sectional snapshot combining 2025 insurance data 
   with each country's **most recent available** World Bank year. This 
   means five countries use actual 2025 macro data, while UAE falls back 
   to 2024 (its latest available year). The actual macro year used is 
   retained as its own column (`macro_data_year`) for full transparency, 
   rather than being dropped or assumed uniform.

This per-country "latest available" approach preserves the most current, 
accurate data wherever it exists, rather than discarding good 2025 data 
for five countries just to force artificial uniformity with the one 
country missing it.

In [5]:
latest_macro = (
    worldbank_macro[worldbank_macro["gdp_usd"].notnull()]
    .sort_values("year")
    .groupby("country")
    .tail(1)
    .copy()
)


latest_macro = latest_macro.rename(columns={"year": "macro_data_year"})


assert latest_macro["country"].duplicated().sum() == 0, "Duplicate countries in latest_macro"
assert len(latest_macro) == 6, f"Expected 6 countries, got {len(latest_macro)}"
print(latest_macro[["country", "macro_data_year"]])

gcc_merged = gcc_insurance.merge(
    latest_macro,
    on="country",
    how="left"
)

gcc_merged

         country  macro_data_year
94           UAE             2024
63         Qatar             2025
31        Kuwait             2025
15       Bahrain             2025
79  Saudi Arabia             2025
47          Oman             2025


,country,year,gwp_usd_billion,penetration_pct,density_usd,nonlife_share_pct,life_share_pct,gcc_share_pct,macro_data_year,gdp_usd,population,gdp_per_capita
0,Saudi Arabia,2025,21.0,1.64,582.1,89.5,10.0,43.2,2025,1.276943e+12,36973555,34536.655546
1,UAE,2025,20.5,3.58,1799.3,82.9,17.1,42.2,2024,5.523249e+11,10986400,50273.512624
2,Qatar,2025,2.5,1.13,789.0,88.0,12.0,5.1,2025,2.155596e+11,2972215,72524.906639
3,Kuwait,2025,2.3,1.45,448.8,91.3,8.7,4.7,2025,1.572090e+11,4865298,32312.311995
4,Oman,2025,1.5,1.42,284.7,86.7,13.3,3.1,2025,1.096048e+11,5494691,19947.396623
5,Bahrain,2025,0.8,1.65,484.0,87.5,10.0,1.6,2025,4.896573e+10,1600366,30596.579490


### Merge Result

All six countries merged successfully with no missing values. Five 
countries use 2025 macro data; UAE uses 2024 (its latest available year, 
since 2025 GDP has not yet been published by World Bank). This is 
reflected transparently in the `macro_data_year` column.

### 5. Quality Checks on Merged Data

In [6]:
print("Shape:", gcc_merged.shape)
print("\nMissing values per column:\n", gcc_merged.isnull().sum())
print("\nDuplicate countries:", gcc_merged["country"].duplicated().sum())


gcc_merged["penetration_check"] = (gcc_merged["gwp_usd_billion"] * 1e9 / gcc_merged["gdp_usd"] * 100).round(2)
gcc_merged[["country", "penetration_pct", "penetration_check"]]

Shape: (6, 12)

Missing values per column:
 country              0
year                 0
gwp_usd_billion      0
penetration_pct      0
density_usd          0
nonlife_share_pct    0
life_share_pct       0
gcc_share_pct        0
macro_data_year      0
gdp_usd              0
population           0
gdp_per_capita       0
dtype: int64

Duplicate countries: 0


,country,penetration_pct,penetration_check
0,Saudi Arabia,1.64,1.64
1,UAE,3.58,3.71
2,Qatar,1.13,1.16
3,Kuwait,1.45,1.46
4,Oman,1.42,1.37
5,Bahrain,1.65,1.63


### Finding: Penetration Cross-Check

Recalculating penetration directly from raw GWP and GDP closely matches 
the report's own stated penetration figures for all six countries (within 
0.05 percentage points for five countries). UAE shows a slightly larger 
gap (3.58% reported vs. 3.71% calculated) expected, since UAE's row 
combines 2025 GWP with 2024 GDP (see `macro_data_year`), while the other 
five countries use matching 2025 figures for both. This validates the 
internal consistency of the merged dataset.

Decision: retain the report's original `penetration_pct` as the primary 
figure (it reflects the source's own methodology, e.g., GDP basis or 
exact reporting date).

### Final Check: Merged Table Statistics

In [7]:
gcc_merged.describe()

,year,gwp_usd_billion,penetration_pct,density_usd,nonlife_share_pct,life_share_pct,gcc_share_pct,macro_data_year,gdp_usd,population,gdp_per_capita,penetration_check
count,6.0,6.000000,6.000000,6.000000,6.000000,6.000000,6.000000,6.000000,6.000000e+00,6.000000e+00,6.000000,6.000000
mean,2025.0,8.100000,1.811667,731.316667,87.650000,11.850000,16.650000,2024.833333,3.934345e+11,1.048209e+07,40031.893819,1.828333
std,0.0,9.818554,0.886734,548.931513,2.840951,3.048114,20.218877,0.408248,4.674384e+11,1.336980e+07,18670.235184,0.938923
min,2025.0,0.800000,1.130000,284.700000,82.900000,8.700000,1.600000,2024.000000,4.896573e+10,1.600366e+06,19947.396623,1.160000
25%,2025.0,1.700000,1.427500,457.600000,86.900000,10.000000,3.500000,2025.000000,1.215058e+11,3.445486e+06,31025.512616,1.392500
50%,2025.0,2.400000,1.545000,533.050000,87.750000,11.000000,4.900000,2025.000000,1.863843e+11,5.179994e+06,33424.483771,1.545000
75%,2025.0,16.000000,1.647500,737.275000,89.125000,12.975000,32.925000,2025.000000,4.681336e+11,9.613473e+06,46339.298355,1.637500
max,2025.0,21.000000,3.580000,1799.300000,91.300000,17.100000,43.200000,2025.000000,1.276943e+12,3.697356e+07,72524.906639,3.710000


### 6. Save Cleaned Data

In [8]:
import os
os.makedirs("data/cleaned", exist_ok=True)

gcc_merged.to_csv("data/cleaned/gcc_merged.csv", index=False)
worldbank_macro.to_csv("data/cleaned/gcc_worldbank_macro_cleaned.csv", index=False)

print("Saved gcc_merged.csv:", gcc_merged.shape)
print("Saved gcc_worldbank_macro_cleaned.csv:", worldbank_macro.shape)

Saved gcc_merged.csv: (6, 13)
Saved gcc_worldbank_macro_cleaned.csv: (96, 5)


Note: `gcc_worldbank_macro_cleaned.csv` retains all 96 rows, including 
UAE's null 2025 GDP the full time series is preserved as-is for future 
time-series work. The null is only excluded when building `gcc_merged` above.

### Summary Data Cleaning Complete

Two cleaned datasets produced:
- `data/cleaned/gcc_merged.csv` 6 rows, 13 columns. Combines 2025 
  insurance data with each country's most recent available macro data 
  (5 countries: 2025; UAE: 2024, tracked in `macro_data_year`). Includes 
  a `penetration_check` column validating reported penetration against 
  raw GWP/GDP.
- `data/cleaned/gcc_worldbank_macro_cleaned.csv` 96 rows, 5 columns. 
  Full 2010–2025 time series, country names standardized, `year` confirmed 
  as integer.

**Next notebook:** `03_uae_data_collection_cleaning.ipynb` — sourcing and 
cleaning UAE-specific historical line-of-business data.